# 18-figures, paper figures on the clean test set

Settings: **Accelerator None (CPU)**, **Internet On**, secret `kaggle_dpfl-braintumor` attached. No dataset input is needed: everything is computed from the run, audit and NCC files already on GitHub.

Runs code tag `v0.8-figures`: tests, then the tables behind the figures (clean test set, D51), then figures 1 to 4 with a comparison against the committed versions, then pushes tables and figures.

Generated by `scripts/make_launcher.py --kind figures`, edit that script rather than this notebook.

In [ ]:
import os, sys, glob, subprocess, datetime, shutil
from kaggle_secrets import UserSecretsClient

USER, EMAIL, REPO = "shashiranjan-jnu", "shashi17_scs@jnu.ac.in", "dpfl-braintumor"
CODE_REF = "v0.8-figures"
token = UserSecretsClient().get_secret("kaggle_dpfl-braintumor")
os.environ["GIT_PUSH_URL"] = f"https://{token}@github.com/{USER}/{REPO}.git"
CLEAN = f"https://github.com/{USER}/{REPO}.git"
CODE, RES = "/tmp/code", "/tmp/results"

def run(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(token, "***")
    if check and r.returncode != 0:
        raise RuntimeError(out[-3000:])
    return out

for d in (CODE, RES):
    shutil.rmtree(d, ignore_errors=True)
    run(["git", "clone", "-q", os.environ["GIT_PUSH_URL"], d])
    run(["git", "remote", "set-url", "origin", CLEAN], cwd=d)
run(["git", "checkout", "-q", CODE_REF], cwd=CODE)
run(["git", "config", "user.name", USER], cwd=RES)
run(["git", "config", "user.email", EMAIL], cwd=RES)
sys.path.insert(0, CODE)
from scripts.run_grid import push_run


print(run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"])[-500:])  # opacus etc.
import torch
TODAY = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d")
COMMIT = run(["git", "rev-parse", "HEAD"], cwd=CODE).strip()
for f in ("results/ncc/curated_pairs_ncc090.csv", "results/tables/week05_h5_mechanism.csv",
          "configs/grids/week07_wn_seeds.yaml"):
    assert os.path.exists(f"{RES}/{f}"), f"missing {f} in the results repository"
n_logits = len(glob.glob(f"{RES}/results/audit/*/logits.npz"))
print("code", CODE_REF, run(["git", "rev-parse", "--short", "HEAD"], cwd=CODE).strip(), "| logits files", n_logits)

In [ ]:
# Test suite on the Kaggle image, report pushed to results/tests/.
sys.path.insert(0, CODE)
from scripts.run_grid import push_run
r = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests", "-p", "no:cacheprovider"],
                   cwd=CODE, capture_output=True, text=True)
report = f"date {TODAY}\ncode {CODE_REF} {COMMIT}\ntorch {torch.__version__}\nreturncode {r.returncode}\n\n" + r.stdout[-6000:]
print(report[-2500:])
os.makedirs(f"{RES}/results/tests", exist_ok=True)
open(f"{RES}/results/tests/{TODAY}_{CODE_REF}_pytest.txt", "w").write(report)
print("push", push_run(RES, "results/tests", f"Test report {CODE_REF}"))
assert r.returncode == 0, "Tests failed, stop here and report"

In [ ]:
# Tables the figures read, recomputed from the run and audit files on GitHub (CPU, a few minutes).
R = f"{RES}/results"
T = f"{R}/tables"
print(run([sys.executable, "scripts/ncc_sensitivity.py", "--results", R, "--out", f"{T}/week07_ncc_sensitivity"], cwd=CODE)[-1500:])
print(run([sys.executable, "scripts/mia_tables.py", "--results", R, "--grids-dir", f"{RES}/configs/grids",
           "--exclude", f"{R}/ncc/contaminated_test_ids.txt", "--out", f"{T}/week07_clean"], cwd=CODE)[-2500:])
print(run([sys.executable, "scripts/xai_tables.py", "--results", R, "--out", f"{T}/week07_xai_border"], cwd=CODE)[-800:])

In [ ]:
# Figures 1 to 4, then a check against the copies already on GitHub, then push.
import pandas as pd, numpy as np
old = {f: pd.read_csv(f"{RES}/figures/figure_data/{f}") for f in
       ("fig1_privacy_utility.csv", "fig2_lira_roc.csv", "fig3_explanations.csv", "fig4_h5_noise.csv")
       if os.path.exists(f"{RES}/figures/figure_data/{f}")}
print(run([sys.executable, "scripts/make_figures.py", "--results", R, "--out", f"{RES}/figures"], cwd=CODE))
for f, o in old.items():
    n = pd.read_csv(f"{RES}/figures/figure_data/{f}")
    num = [c for c in o.columns if o[c].dtype.kind == "f" and c in n]
    diff = max(float(np.nanmax(np.abs(o[c].values - n[c].values))) for c in num) if len(o) == len(n) else float("inf")
    print(f"{f}: {len(n)} rows, largest difference from the committed version {diff:.2e}")
from IPython.display import Image, display
for f in ("fig1_privacy_utility", "fig2_lira_roc", "fig3_explanations", "fig4_h5_noise"):
    display(Image(f"{RES}/figures/{f}.png", width=900))
print("push tables", push_run(RES, "results/tables", f"Tables regenerated on Kaggle ({CODE_REF})"))
print("push figures", push_run(RES, "figures", f"Figures regenerated on Kaggle ({CODE_REF})"))